In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
df = pd.read_csv(ROOT/"data_processed"/"clinical_longitudinal.csv")

# Keep rows with needed columns & drop if severely incomplete
X_cols = ["AGE","SEX","NP3TOT","MCATOT","NP1TOT","NP2TOT"]
keep = df[["PATNO","EVENT_ID","Y_PD"] + X_cols].copy()

# Convert numeric
for c in X_cols:
    keep[c] = pd.to_numeric(keep[c], errors="coerce")

# Drop rows missing *all* core features
keep = keep.dropna(subset=["AGE","NP3TOT","MCATOT","NP1TOT","NP2TOT"], how="all")

print("Visits:", len(keep))
print("Features:", X_cols)
print("Label counts:\n", keep["Y_PD"].value_counts())

Visits: 13012
Features: ['AGE', 'SEX', 'NP3TOT', 'MCATOT', 'NP1TOT', 'NP2TOT']
Label counts:
 Y_PD
0    7107
1    5905
Name: count, dtype: int64


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

# Prepare X, y (clean + impute)
X = keep[X_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
y = keep["Y_PD"].astype(int).clip(0,1)

def build_model():
    return XGBClassifier(
        objective="binary:logistic",
        base_score=0.5,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

# Normal train-test split if 2 classes exist
if y.nunique() == 2 and y.value_counts().min() >= 10:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    model = build_model().fit(X_tr, y_tr)
    y_hat = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]

    print("Classification report:\n", classification_report(y_te, y_hat))
    try:
        print("ROC-AUC:", roc_auc_score(y_te, y_prob))
    except:
        print("ROC-AUC cannot be computed")
    print("Confusion matrix:\n", confusion_matrix(y_te, y_hat))

else:
    print("⚠️ Dataset imbalance too high - fallback to CV.")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model = build_model().fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_te)[:,1]
        try:
            aucs.append(roc_auc_score(y_te, y_prob))
        except:
            pass
    print("CV ROC-AUC:", np.mean(aucs))

Classification report:
               precision    recall  f1-score   support

           0       0.89      0.85      0.87      1422
           1       0.83      0.87      0.85      1181

    accuracy                           0.86      2603
   macro avg       0.86      0.86      0.86      2603
weighted avg       0.86      0.86      0.86      2603

ROC-AUC: 0.9364665097041649
Confusion matrix:
 [[1210  212]
 [ 149 1032]]
